# PPO Evaluation Notebook

This notebook allows you to load a trained model and visualize its actions and PnL on a specific validation slice.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('.')

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from core.experiment_manager import get_experiment_paths
from core.data.data_loader import load_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v6 import TradingEnvV6
from agents.ppo_agent import create_ppo_agent
from core.config import VAL_SLICES, N_STACK

### 1. Configuration
Specify your experiment name, the type of model you want to evaluate (e.g. `transformer`, `gru`, `cnn`, `mlp`), and the exact validation slice you want to look at.

In [ ]:
EXP_NAME = "ppo_purged_v6_multi"
EXTRACTOR = "transformer" # 'mlp', 'cnn', 'gru', 'transformer'
SYMBOL = "BTCUSDT"
VAL_SLICE_NAME = "bull_1"

model_path, norm_path, _ = get_experiment_paths(EXP_NAME)

### 2. Load Data and Extract Features

In [ ]:
fg = FeatureGenerator()
df = load_crypto_data(SYMBOL, start_date="2020-01-01", end_date="2026-06-26", interval="4h", use_cache=True)
processed_df = fg.transform(df)

start, end = VAL_SLICES[VAL_SLICE_NAME]
val_df = processed_df.loc[start:end].reset_index(drop=True)
print(f"Loaded {len(val_df)} candles for {VAL_SLICE_NAME}")

### 3. Create Environment and Load Model

In [ ]:
def make_env():
    return TradingEnvV6(df=val_df, domain_randomization=False, t_max=None)

env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=N_STACK)
env = VecNormalize.load(norm_path, env)
env.training = False
env.norm_reward = False

hmm_cols = [c for c in val_df.columns if 'hmm_regime' in c]
num_hmm_states = len(hmm_cols)

model = create_ppo_agent(
    env, 
    extractor_type=EXTRACTOR, 
    num_hmm_states=num_hmm_states, 
    n_stack=N_STACK
)

# Override initialized weights with the saved trained weights
trained_model = PPO.load(model_path)
model.policy.load_state_dict(trained_model.policy.state_dict())
print("Model weights loaded successfully!")

### 4. Run Evaluation and Plot Results

In [ ]:
obs = env.reset()
done = False

pnls = []
actions = []

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, dones, infos = env.step(action)
    done = dones[0]
    
    actions.append(action[0][0])
    pnls.append(infos[0]['pnl'])

closes = val_df['close'].iloc[:len(pnls)].values

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

ax1.plot(closes, label="Price", color="blue")
ax1.set_title(f"Price for {SYMBOL} ({VAL_SLICE_NAME})")
ax1.set_ylabel("Price")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(actions, label="Target Weight (Action)", color="orange")
ax2.axhline(0, color='red', linestyle='--')
ax2.axhline(1, color='gray', linestyle=':')
ax2.axhline(-1, color='gray', linestyle=':')
ax2.set_title("Agent Actions (-1 Short, 0 Cash, 1 Long)")
ax2.set_ylabel("Weight")
ax2.legend()
ax2.grid(alpha=0.3)

ax3.plot(pnls, label="Agent PnL %", color="green")
ax3.axhline(0, color='gray', linestyle='--')
ax3.set_title("Cumulative PnL %")
ax3.set_ylabel("PnL %")
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final PnL: {pnls[-1]:.2f}%")
print(f"Max Drawdown: {infos[0]['drawdown']:.2f}%")
print(f"Sortino Proxy: {infos[0]['episode_sortino']:.4f}")